In [2]:
!pip install -q requests pandas torch transformers scikit-learn

import requests
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification

"""
function that will return: list of profressor names + list of all their reviews
params : target - # of profs we want (5), min_reviews - each prof must have >= 5 reviews to be used, limit - ask for 100 at a time (when calling professors endpoint),
max_batches - safety cap so we don't loop forever
"""
def get_professors_with_reviews(target=5, min_reviews=5, limit=100, max_batches=50):
  offset = 0 # where in the list of professors to start
  selected_profs = [] # empty list where we’ll store the names of the 5 professors
  review_rows = [] # empty list where we’ll store all the reviews (each review = one row)

  while len(selected_profs) < target:
    # keep looping until we haven't found enough profs
    url = f"https://planetterp.com/api/v1/professors?limit={limit}&offset={offset}"
    response = requests.get(url)

    if response.status_code != 200:
        print("Failed to fetch professors:", response.status_code)
        break

    batch = response.json()
    if not batch:
        print("No more professors returned from API.")
        break

    print(f"\nChecking professor batch starting at offset {offset}...")


    offset += limit # next page of profs
  return selected_profs, review_rows

  for prof in batch:
      if len(selected_profs) >= target: #if we alr have 5, break
          break

      name = prof["name"]

        # Skip if already selected
      if name in selected_profs:
          continue

      prof_url = f"https://planetterp.com/api/v1/professor?name={name}&reviews=true"
      r = requests.get(prof_url)

      if r.status_code != 200:
          print(f"  Skipping {name} (failed to fetch reviews: {r.status_code})")
          continue

      prof_data = r.json()
      reviews = prof_data.get("reviews", [])

        # keep only reviews that have both text and rating
      usable_reviews = []
      for review in reviews:
          text = review.get("review", "")
          rating = review.get("rating", None)
          if not text or rating is None:
              continue
          usable_reviews.append(review)

      if len(usable_reviews) < min_reviews:
          print(f"  Skipping {name}: only {len(usable_reviews)} usable reviews")
          continue

        # This professor is good enough
      print(f"  **Using {name}: {len(usable_reviews)} usable reviews**")
      selected_profs.append(name)

      for review in usable_reviews:
          review_rows.append({
              "professor": name,
              "course": review.get("course", None),
              "review": review.get("review", ""),
              "rating": review.get("rating", None),
              "expected_grade": review.get("expected_grade", None),
              "created": review.get("created", None),
          })


        offset += limit
        batches_checked += 1

    return selected_profs, review_rows


# Run the collection (you already saw similar output)
TARGET_PROFS = 5
MIN_REVIEWS_PER_PROF = 5

selected_profs, review_rows = get_professors_with_reviews(
    target=TARGET_PROFS,
    min_reviews=MIN_REVIEWS_PER_PROF
)

print("\nSelected professors:")
for name in selected_profs:
    print(" -", name)

print(f"\nTotal reviews collected: {len(review_rows)}")

df = pd.DataFrame(review_rows)

# Clean a bit: drop rows with missing rating or empty review
df = df[
    df["rating"].notna()
    & df["review"].notna()
    & (df["review"].str.strip() != "")
]

print("\nDataset sample:")
print(df.head())
print(f"\nFinal dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")

# Save to CSV to show how data was prepared
df.to_csv("professor_review_dataset.csv", index=False)
print("\nSaved dataset to 'professor_review_dataset.csv'")


# ============================================================
# 2. Prepare data for modeling (review text + star rating)
# ============================================================

data = df[["review", "rating"]].copy()
data["rating"] = data["rating"].astype(int)

# Keep only ratings between 1 and 5
data = data[(data["rating"] >= 1) & (data["rating"] <= 5)]

# Map ratings 1–5 -> labels 0–4 (what the model predicts)
data["label"] = data["rating"] - 1

# Simple train/val split
if data["label"].nunique() > 1 and len(data) > 5:
    stratify_labels = data["label"]
else:
    stratify_labels = None

train_df, val_df = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=stratify_labels
)

print("\nTrain size:", len(train_df), "Val size:", len(val_df))


# ============================================================
# 3. Build a Dataset and DataLoader (plain PyTorch)
# ============================================================

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class ReviewDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, "review"]
        label = int(self.df.loc[idx, "label"])

        encoded = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }
        return item

train_dataset = ReviewDataset(train_df, tokenizer)
val_dataset = ReviewDataset(val_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)


# ============================================================
# 4. Define model + simple training loop
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing device:", device)

num_labels = 5  # ratings 1–5

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

def train_one_epoch(model, data_loader, optimizer, device):
    model.train()
    total_loss = 0.0

    for batch in data_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(data_loader) if len(data_loader) > 0 else 0.0
    return avg_loss

def evaluate(model, data_loader, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(data_loader) if len(data_loader) > 0 else 0.0
    accuracy = correct / total if total > 0 else 0.0
    return avg_loss, accuracy


# ============================================================
# 5. Train for a few epochs
# ============================================================

EPOCHS = 3

for epoch in range(EPOCHS):
    print(f"\n===== Epoch {epoch + 1}/{EPOCHS} =====")
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, device)

    print(f"Train loss: {train_loss:.4f}")
    print(f"Val   loss: {val_loss:.4f} | Val accuracy: {val_acc:.4f}")


# ============================================================
# 6. Make some example predictions
# ============================================================

def predict_rating(text, model, tokenizer, device):
    model.eval()
    with torch.no_grad():
        encoded = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=256,
            return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        outputs = model(**encoded)
        logits = outputs.logits
        pred_label = int(torch.argmax(logits, dim=-1).item())
        # map back from 0–4 -> 1–5 stars
        return pred_label + 1

print("\nSome validation examples (true vs predicted):")
for i in range(min(5, len(val_df))):
    row = val_df.iloc[i]
    text = row["review"]
    true_rating = int(row["rating"])
    pred_rating = predict_rating(text, model, tokenizer, device)

    print("\n---")
    print("Review:", text[:200].replace("\n", " "), "...")
    print("True rating:     ", true_rating)
    print("Predicted rating:", pred_rating)

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 88)